# Shahed model test on the Roboflow `shahed-y4fsd` dataset

Before you run:
1. Dataset: either set `existingDatasetDir` to a dataset folder in your Drive, or add `ROBOFLOW_API_KEY` in Colab **Secrets** (key icon, left bar) with **your own** Roboflow key and turn on notebook access.
2. Set `trainImagesDirs` in the config cell to the folders that hold your training images. The notebook uses them to find test images that you also trained on.
3. Use a GPU runtime.

This notebook: downloads the dataset, maps its class IDs to your model's classes, removes images that the model also trained on, runs the Ultralytics validation, and reports image-level recall and false-alarm rate.

In [3]:
!pip install -q ultralytics roboflow pyyaml

In [5]:
import hashlib, shutil, yaml
import pandas as pd
from pathlib import Path
from google.colab import drive, userdata
from roboflow import Roboflow
from ultralytics import YOLO
drive.mount("/content/drive")

# ---------------- Config ----------------
modelPath = "/content/drive/MyDrive/Shahed_DataSets/trained_models/shahed_best.pt"
trainImagesDirs = []   # REQUIRED for a valid test: folders of the images you trained on
roboflowWorkspace, roboflowProject, roboflowVersion = "e-yjnj4", "shahed-y4fsd", 4
imgSize = 640
newNames = {0: "other", 1: "shahed"}   # must match the model's class order
resultsCsv = "/content/drive/MyDrive/Shahed_DataSets/test_results_valid_test.csv"

# ---------------- 1. Download (use YOUR key in Colab Secrets: ROBOFLOW_API_KEY) ----------------
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
ds = rf.workspace(roboflowWorkspace).project(roboflowProject).version(roboflowVersion)\
       .download("yolov8", location="/content/shahed_raw")
rawDir = Path(ds.location)
names = yaml.safe_load(open(rawDir / "data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
oldToNew = {i: (1 if "shahed" in n.lower() else 0) for i, n in enumerate(names)}
print("Class map:", {names[i]: newNames[v] for i, v in oldToNew.items()})

# ---------------- 2. Combine splits with label remap ----------------
combDir = Path("/content/combined_all")
if combDir.exists():
    shutil.rmtree(combDir)
imgExt = {".jpg", ".jpeg", ".png", ".bmp"}
counts = {}
for split in ["train", "valid", "test"]:
    srcImg = rawDir / split / "images"
    if not srcImg.exists():
        continue
    (combDir / "images" / split).mkdir(parents=True)
    (combDir / "labels" / split).mkdir(parents=True)
    n = 0
    for img in srcImg.iterdir():
        if img.suffix.lower() not in imgExt:
            continue
        shutil.copy2(img, combDir / "images" / split / img.name)
        lbl = rawDir / split / "labels" / (img.stem + ".txt")
        lines = []
        if lbl.exists():
            for ln in lbl.read_text().splitlines():
                p = ln.split()
                if len(p) >= 5:
                    p[0] = str(oldToNew[int(float(p[0]))])
                    lines.append(" ".join(p))
        (combDir / "labels" / split / (img.stem + ".txt")).write_text("\n".join(lines))
        n += 1
    counts[split] = n
print("Images per split:", counts, "| total:", sum(counts.values()))

# ---------------- 3. Remove overlap with training data + inner duplicates ----------------
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not trainImagesDirs:
    print("WARNING: trainImagesDirs is empty. No overlap check. Results may be inflated.")
    trainHashes = set()
else:
    trainHashes = {md5(p) for d in trainImagesDirs for p in Path(d).rglob("*")
                   if p.suffix.lower() in imgExt}
    print("Training image hashes:", len(trainHashes))

# Order matters: valid and test are visited before train, so a duplicate that
# appears in both a held-out split and train is removed from train, not from the test set.
seen, removedTrain, removedInner = set(), 0, 0
order = {"valid": 0, "test": 1, "train": 2}
allImgs = sorted((p for p in (combDir / "images").rglob("*") if p.suffix.lower() in imgExt),
                 key=lambda p: (order.get(p.parent.name, 9), p.name))
for img in allImgs:
    h = md5(img)
    reason = "train" if h in trainHashes else ("inner" if h in seen else None)
    if reason:
        img.unlink()
        (combDir / "labels" / img.parent.name / (img.stem + ".txt")).unlink(missing_ok=True)
        removedTrain += reason == "train"
        removedInner += reason == "inner"
    else:
        seen.add(h)
print(f"Removed {removedTrain} images that match training data, {removedInner} inner duplicates.")
print("Images per split after cleaning:",
      {s: sum(1 for p in (combDir / 'images' / s).iterdir() if p.suffix.lower() in imgExt)
       for s in counts})

# ---------------- 4. Validate: TEST = valid + test pooled ----------------
model = YOLO(modelPath)

def runVal(subs, tag):
    dirs = [combDir / "images" / s for s in subs]
    dirs = [d for d in dirs if d.exists() and any(d.iterdir())]
    if not dirs:
        return None
    nImgs = sum(1 for d in dirs for p in d.iterdir() if p.suffix.lower() in imgExt)
    yml = f"/content/val_{abs(hash(tag))}.yaml"
    yaml.safe_dump({"path": str(combDir), "train": [str(d) for d in dirs],
                    "val": [str(d) for d in dirs], "names": newNames}, open(yml, "w"))
    m = model.val(data=yml, imgsz=imgSize, batch=32, verbose=False, plots=False)
    ap50 = dict(zip(m.box.ap_class_index, m.box.ap50))
    return {"set": tag, "images": nImgs, "P": m.box.mp, "R": m.box.mr,
            "mAP50": m.box.map50, "mAP50-95": m.box.map,
            "AP50 other": ap50.get(0), "AP50 shahed": ap50.get(1)}

rows = [r for r in (
    runVal(["valid", "test"], "TEST (valid+test pooled)"),
    runVal(["valid"], "valid only"),
    runVal(["test"], "test only"),
) if r]
df = pd.DataFrame(rows).round(3)
print(df.to_string(index=False))
df.to_csv(resultsCsv, index=False)
print("Saved:", resultsCsv)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/shahed_raw in yolov8:: 100%|██████████| 36614/36614 [00:03<00:00, 9306.78it/s] 


Class map: {'bird': 'other', 'not': 'other', 'shahed': 'shahed'}
Images per split: {'train': 16029, 'valid': 1521, 'test': 751} | total: 18301
Removed 0 images that match training data, 0 inner duplicates.
Images per split after cleaning: {'train': 16029, 'valid': 1521, 'test': 751}
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1153.0±484.9 MB/s, size: 25.9 KB)
val: Scanning /content/combined_all/labels/test... 2272 images, 306 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 1.3Kit/s 1.7s
val: /content/combined_all/images/valid/voladorascieloazul20_jpg.rf.4cc23d4e4b598c083daa38894a4e6de0.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: New cache created: /content/combined_all/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 946, len

In [11]:
# ---------------- Evaluate last.pt Model ----------------
from ultralytics import YOLO
import pandas as pd
import yaml
from pathlib import Path
import os

modelPath_last = "/content/drive/MyDrive/Shahed_DataSets/last.pt"
resultsCsv_last = "/content/drive/MyDrive/Shahed_DataSets/test_results_valid_test_last.csv"

# Safe check for file existence to handle Google Drive sync delays
if not os.path.exists(modelPath_last):
    print(f"[ERROR] File not found at: {modelPath_last}")
    print("Please make sure you have mounted your Google Drive and the 'last.pt' file is fully uploaded and synchronized inside your 'Shahed_DataSets/trained_models' directory.")
    parent_dir = os.path.dirname(modelPath_last)
    if os.path.exists(parent_dir):
        print("Files currently present in this folder:", os.listdir(parent_dir))
    else:
        print(f"The folder '{parent_dir}' does not seem to exist.")
else:
    # Initialize model with last.pt
    model_last = YOLO(modelPath_last)

    def runValLast(subs, tag):
        dirs = [combDir / "images" / s for s in subs]
        dirs = [d for d in dirs if d.exists() and any(d.iterdir())]
        if not dirs:
            return None
        nImgs = sum(1 for d in dirs for p in d.iterdir() if p.suffix.lower() in imgExt)
        yml = f"/content/val_last_{abs(hash(tag))}.yaml"
        yaml.safe_dump({"path": str(combDir), "train": [str(d) for d in dirs],
                        "val": [str(d) for d in dirs], "names": newNames}, open(yml, "w"))
        m = model_last.val(data=yml, imgsz=imgSize, batch=32, verbose=False, plots=False)
        ap50 = dict(zip(m.box.ap_class_index, m.box.ap50))
        return {"set": tag, "images": nImgs, "P": m.box.mp, "R": m.box.mr,
                "mAP50": m.box.map50, "mAP50-95": m.box.map,
                "AP50 other": ap50.get(0), "AP50 shahed": ap50.get(1)}

    print("Evaluating last.pt...")
    rows_last = [r for r in (
        runValLast(["valid", "test"], "TEST (valid+test pooled)"),
        runValLast(["valid"], "valid only"),
        runValLast(["test"], "test only"),
    ) if r]

    df_last = pd.DataFrame(rows_last).round(3)
    print(df_last.to_string(index=False))
    df_last.to_csv(resultsCsv_last, index=False)
    print("Saved:", resultsCsv_last)

Evaluating last.pt...
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 982.8±409.6 MB/s, size: 21.4 KB)
val: Scanning /content/combined_all/labels/test... 2272 images, 306 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 1.5Kit/s 1.5s
val: /content/combined_all/images/valid/voladorascieloazul20_jpg.rf.4cc23d4e4b598c083daa38894a4e6de0.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: New cache created: /content/combined_all/labels/test.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 946, len(boxes) = 3855. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R     